In [13]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

# Define your CodeBERTMultiTaskClassifier class here
class CodeBERTMultiTaskClassifier(nn.Module):
    def __init__(self, num_bug_types):  # Add num_bug_types as a parameter
        super(CodeBERTMultiTaskClassifier, self).__init__()
        self.model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2, output_hidden_states=True)
        self.bug_type_classifier = nn.Linear(self.model.config.hidden_size, num_bug_types)  # Ensure num_bug_types is defined

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask)

        # Use the [CLS] token for binary classification
        logits_binary = outputs.logits
        
        # Use the [CLS] token for bug type classification
        bug_type_logits = self.bug_type_classifier(outputs.hidden_states[-1][:, 0, :])
        
        return logits_binary, bug_type_logits

# Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

# Define the model and load the saved state_dict
num_bug_types = 6  # Assuming you have 6 bug types
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CodeBERTMultiTaskClassifier(num_bug_types).to(device)
model.load_state_dict(torch.load("C:/Users/ABHIMANYU/Downloads/codebert_classifierbugtype(500)0.615819209039548.pth"))  # Update with your path
model.eval()  # Set the model to evaluation mode

# Function to predict code type and bug type
def predict_code_and_bug_type(code_sample):
    # Tokenize the input
    tokenized_input = tokenizer(
        code_sample,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )
    
    input_ids = tokenized_input['input_ids'].to(device)
    attention_mask = tokenized_input['attention_mask'].to(device)

    with torch.no_grad():  # No need to calculate gradients during inference
        logits_binary, logits_bug_type = model(input_ids, attention_mask)

    # Task 1: Predict whether the code is AI-generated or human-written
    probabilities = torch.nn.functional.softmax(logits_binary, dim=1)  # Apply softmax to get probabilities
    ai_generated_prob = probabilities[0][1].item() * 100  # Probability of AI-generated
    human_written_prob = probabilities[0][0].item() * 100  # Probability of human-written

    print(f"Code is {ai_generated_prob:.3f}% AI-generated")
    print(f"Code is {human_written_prob:.3f}% human-written")

    # Task 2: Predict bug type for human-written code
    if torch.argmax(logits_binary, dim=1).item() == 0:  # If human-written
        bug_type_logits = logits_bug_type[0]
        bug_type_probabilities = torch.nn.functional.softmax(bug_type_logits, dim=0)
        bug_type_prediction = torch.argmax(bug_type_probabilities).item()
        
        # Assuming you have a mapping of bug types to strings
        bug_type_mapping_inv = {
            0: 'missing logic',
            1: 'excess logic',
            2: 'value misuse',
            3: 'operator misuse',
            4: 'variable misuse',
            5: 'function misuse'
        }
        predicted_bug_type = bug_type_mapping_inv[bug_type_prediction]
        print(f"Predicted Bug Type: {predicted_bug_type}")
    else:
        print("No bug type prediction for AI-generated code.")

# Example usage
sample_code =[
    '''

    '''
]  # Replace with your code sample
predict_code_and_bug_type(sample_code)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Code is 50.513% AI-generated
Code is 49.487% human-written
No bug type prediction for AI-generated code.
